# Section 8 — Trapezoidal and Simpson's Rules

[← Riemann Sums](07_riemann_sums.ipynb) ·
[Module 2](README.md)

The integration methods in this unit can be viewed as increasingly accurate
local polynomial approximations:

- Riemann sums: piecewise constants;
- trapezoidal rule: piecewise linear functions;
- Simpson's rule: piecewise quadratics.

This is the same progression emphasized in Sullivan's numerical-calculus
chapter.

## Trapezoidal rule

On one interval $[a,b]$ with $h=b-a$,

$$
\int_a^b f(x)\,dx
\approx
\frac{h}{2}[f(a)+f(b)].
$$

For the composite rule with $n$ equal subintervals,

$$
T_n
=
\frac{h}{2}
\left[
f(x_0)
+2\sum_{j=1}^{n-1}f(x_j)
+f(x_n)
\right].
$$

Its global error is second order:

$$
E_T=O(h^2).
$$

## Simpson's rule

Using two subintervals and three points,

$$
\int_a^b f(x)\,dx
\approx
\frac{h}{3}
\left[
f(x_0)+4f(x_1)+f(x_2)
\right],
$$

where now $h=(b-a)/2$.

For composite Simpson with even $n$,

$$
S_n=
\frac{h}{3}
\left[
f(x_0)
+4\sum_{\text{odd }j}f(x_j)
+2\sum_{\text{even }j}f(x_j)
+f(x_n)
\right].
$$

Its global error is fourth order:

$$
E_S=O(h^4).
$$

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def composite_trapezoid(function, a, b, n):
    """Approximate an integral with composite trapezoids."""
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    total = function(x[0]) + function(x[-1])
    total += 2.0 * np.sum(function(x[1:-1]))
    return 0.5 * h * total


def composite_simpson(function, a, b, n):
    """Approximate an integral with composite Simpson's rule."""
    if n % 2 != 0:
        raise ValueError("Simpson's rule requires even n.")

    h = (b - a) / n
    x = np.linspace(a, b, n + 1)

    total = function(x[0]) + function(x[-1])
    total += 4.0 * np.sum(function(x[1:-1:2]))
    total += 2.0 * np.sum(function(x[2:-1:2]))

    return (h / 3.0) * total

## Trapezoidal-rule example from the course

Approximate

$$
\int_0^3 \sin(2x)\,dx
$$

with one trapezoid.

The single-interval error formula is

$$
\int_a^b f(x)\,dx
=
\frac{h}{2}[f(a)+f(b)]
-\frac{h^3}{12}f''(\xi).
$$

For $f(x)=\sin(2x)$,

$$
f''(x)=-4\sin(2x),
$$

so $|f''(x)|\le4$.

In [ ]:
def sine_function(x):
    return np.sin(2.0 * x)


a = 0.0
b = 3.0
h = b - a

trap_one = 0.5 * h * (
    sine_function(a) + sine_function(b)
)
exact = 0.5 * (1.0 - math.cos(6.0))
trap_error = abs(exact - trap_one)
trap_bound = h**3 * 4.0 / 12.0

print(f"trapezoid approximation = {trap_one:.10f}")
print(f"exact value             = {exact:.10f}")
print(f"actual error            = {trap_error:.10f}")
print(f"error bound             = {trap_bound:.10f}")

## Composite Simpson example from the course

Approximate

$$
\int_1^2 \frac{1}{x}\,dx
$$

with composite Simpson's rule using $n=4$ subintervals, then compare with the
exact value $\ln 2$.

In [ ]:
def reciprocal(x):
    return 1.0 / x


simpson_value = composite_simpson(reciprocal, 1.0, 2.0, 4)
exact_log = math.log(2.0)
simpson_error = abs(exact_log - simpson_value)

print(f"Simpson approximation = {simpson_value:.10f}")
print(f"exact ln(2)           = {exact_log:.10f}")
print(f"actual error          = {simpson_error:.3e}")

## Compare all integration methods

For a smooth function, we expect approximately:

| Method | Typical global order |
|---|---:|
| Left/right Riemann | 1 |
| Midpoint | 2 |
| Trapezoidal | 2 |
| Simpson | 4 |

The experiment below compares midpoint, trapezoidal, and Simpson approximations
for

$$
\int_0^1 e^x\,dx=e-1.
$$

In [ ]:
def midpoint_rule(function, a, b, n):
    h = (b - a) / n
    x = a + h * (np.arange(n) + 0.5)
    return h * np.sum(function(x))


def exp_function(x):
    return np.exp(x)


exact_exp = math.e - 1.0
n_values = 2 ** np.arange(1, 9)

mid_errors = []
trap_errors = []
simp_errors = []

for n in n_values:
    mid_errors.append(
        abs(exact_exp - midpoint_rule(exp_function, 0.0, 1.0, n))
    )
    trap_errors.append(
        abs(
            exact_exp
            - composite_trapezoid(exp_function, 0.0, 1.0, n)
        )
    )
    simp_errors.append(
        abs(
            exact_exp
            - composite_simpson(exp_function, 0.0, 1.0, n)
        )
    )

plt.figure(figsize=(9, 5))
plt.loglog(n_values, mid_errors, "o-", label="midpoint")
plt.loglog(n_values, trap_errors, "o-", label="trapezoidal")
plt.loglog(n_values, simp_errors, "o-", label="Simpson")
plt.xlabel("number of subintervals n")
plt.ylabel("absolute error")
plt.title("Composite Integration Methods")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Choosing $n$ from an error bound

For the composite trapezoidal rule,

$$
|E_T|
\le
\frac{b-a}{12}h^2
\max_{[a,b]}|f''(x)|.
$$

For composite Simpson's rule,

$$
|E_S|
\le
\frac{b-a}{180}h^4
\max_{[a,b]}|f^{(4)}(x)|.
$$

These formulas let us choose a sufficient number of subintervals **before**
performing the approximation.

### References

- Eric Sullivan, *Numerical Methods: An Inquiry-Based Approach with Python*.
- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 course lecture notes and review materials.


[← Module 2](README.md)